In [44]:
import torch
import torch.nn as nn
import numpy as np
from time import time
from models import get_model
import os
import random
from dataset import get_data
from environment import Env
from time import time
from torch.utils.data import WeightedRandomSampler
import matplotlib.pyplot as plt
from agent import Agent
from inference import Inference
from environment import Env
import seaborn as sns
from math import ceil
from tqdm import tqdm
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [45]:
class arg:
    def __init__(self) -> None:
        pass

In [46]:
args = arg()
args.random_seed = int(17)
args.model = 'simple'
args.dataset = 'simu_data'
args.task_id = int(2)
args.data_type = str('dependent_dependent_simple')

In [47]:
args.train_lr = 0.003
args.val_test_split = [0.25,0.25]
args.n_feature = int(58) if args.dataset == 'ACIC2016' else int(25)
args.missing_ratio = float(1)
args.disable_cuda = False
args.complete = False
args.pretrain = int(1000)
args.pretrain_sample = str('both')
args.mode = str('double')
args.decay = float(0.999)
args.gamma = float(1)
args.dropout = False
args.batchnorm = False
args.done_action_train = False
args.p = float(0)
args.group_norm = float(0)
args.save_dir = str('result')
args.inf_hidden_sizes = [512,512]
args.policy_hidden_sizes = [32]
args.shared_dim = int(16)
args.target_update_freq = int(100)
args.eps_start = float(1.)
args.eps_end = float(0.1)
args.decay_rate = float(2)
args.n_env = int(32)
args.nsteps = int(4)
args.normalize = True
args.embedded_dim = int(16)
args.lstm_size = int(16)
args.n_shuffle = int(5)
args.r_cost = float(1.0)
args.cost_from_file = False
args.batch_size = int(128)
args.message = str('')
args.buffer_size = int(10000)
random.seed(args.random_seed)
np.random.seed(args.random_seed)
torch.manual_seed(args.random_seed)
if not args.disable_cuda and torch.cuda.is_available():
    args.device = torch.device('cuda')
    torch.cuda.manual_seed(args.random_seed)
else:
    args.device = torch.device('cpu')
args.save_dir = os.path.join(os.getcwd(),args.save_dir)
if args.dataset == 'simu_data':
    args.save_path = args.dataset + '_' + args.data_type
elif args.dataset == 'ACIC2016':
    args.save_path = args.dataset + '_' + str(args.task_id)
else:
    args.save_path = args.dataset
args.save_path = args.save_path + '_cost{}'.format(args.r_cost)
args.save_path = os.path.join(args.save_dir, args.save_path)
args.csv_path = args.save_path
args.save_path = args.save_path + '_seed{}'.format(args.random_seed)
args.data_path = os.path.join(os.getcwd(),'dataset')
if not os.path.exists(args.save_path):
    os.makedirs(args.save_path)


In [48]:
class samples_buffer:
    def __init__(self,capacity) -> None:
        self.capacity = capacity
        self.buffer = torch.empty(0)
        self.counter = 0
    
    def push(self,new_data):
        new_data = new_data.to(self.buffer.device)
        if self.counter != 0:
            assert self.buffer.shape[1:] == new_data.shape[1:], f"input dim {new_data.shape[1:]} must be same as recorded dim {self.buffer.shape[1:]}"
        n_new_data = new_data.shape[0]
        if n_new_data == 0:
            return
        if self.counter + n_new_data <= self.capacity: 
            self.buffer = torch.cat([self.buffer,new_data],dim=0)
            self.counter = self.counter + n_new_data
        else:
            self.buffer = torch.cat([self.buffer,new_data],dim=0)
            if self.counter < self.capacity:
                n_drop = self.counter + n_new_data - self.capacity
                prob = torch.cat([n_drop*torch.ones(self.buffer.shape[0])],dim=0)
                drop_ind = WeightedRandomSampler(prob,n_drop,replacement=False)
            else :
                prob = torch.cat([n_new_data*torch.ones(self.capacity),(self.counter - self.capacity)*torch.ones(n_new_data)],dim=0)
                drop_ind = WeightedRandomSampler(prob,n_new_data,replacement=False)
            
            remain_ind = torch.tensor(list(set(range(self.buffer.shape[0])) - set(drop_ind)))
            self.buffer = self.buffer[remain_ind]
            self.counter = self.counter + n_new_data
            
    def sample(self,num):
        if self.counter > self.capacity:
            index = random.sample(list(torch.arange(self.capacity)),num)
        else:
            index = random.sample(list(torch.arange(self.counter)),num)
        return self.buffer[index]

In [49]:
args.X_mode,args.T_mode,args.Y_mode = args.data_type.split('_')
traindata,testdata,valdata = get_data(args)

In [50]:
model = get_model(args)
inf = Inference(model,'T_mode',args,5000)
agent = Agent(model,args,5000)
train_env = Env(args.n_env,traindata,model,args.r_cost)
val_env = Env(args.n_env,valdata,model,args.r_cost)
test_env = Env(args.n_env,testdata,model,args.r_cost)
args.missing_ratio = float(1)
inf.pretrain(traindata,valdata,args,25000,128)
args.missing_ratio = float(1)
inf.test(testdata,args)

start_pretrain
epoch: 10  train_loss: 925.4708251953125  val_loss: 843.6381225585938
epoch: 20  train_loss: 321.77471923828125  val_loss: 282.2826232910156
epoch: 30  train_loss: 25.743627548217773  val_loss: 28.181106567382812
epoch: 40  train_loss: 22.310667037963867  val_loss: 27.020156860351562
epoch: 300  train_loss: 21.949438095092773  val_loss: 27.014110565185547
epoch: 340  train_loss: 28.19824981689453  val_loss: 26.99689292907715
epoch: 450  train_loss: 22.26066017150879  val_loss: 26.99184799194336
epoch: 1270  train_loss: 26.852310180664062  val_loss: 26.991472244262695
epoch: 2970  train_loss: 27.690427780151367  val_loss: 26.990032196044922
epoch: 3310  train_loss: 27.589496612548828  val_loss: 26.98999786376953
pretrain_time: 84.16680240631104
start_inference_test
finish_inference_test
time_use: 0.01970672607421875
mse of tau: tensor(2.0520, device='cuda:0', grad_fn=<DivBackward0>)
mse of y_fact: tensor(24.6087, device='cuda:0', grad_fn=<DivBackward0>)


(tensor(2.0520, device='cuda:0', grad_fn=<DivBackward0>),
 tensor(24.6087, device='cuda:0', grad_fn=<DivBackward0>))

In [51]:
model = get_model(args)
inf = Inference(model,'T_mode',args,5000)
agent = Agent(model,args,5000)
train_env = Env(args.n_env,traindata,model,args.r_cost)
val_env = Env(args.n_env,valdata,model,args.r_cost)
test_env = Env(args.n_env,testdata,model,args.r_cost)
args.missing_ratio = float(0)
inf.pretrain(traindata,valdata,args,25000,128)
args.missing_ratio = float(0)
inf.test(testdata,args)

start_pretrain
epoch: 10  train_loss: 441.0645751953125  val_loss: 222.45787048339844
epoch: 20  train_loss: 109.43637084960938  val_loss: 126.30531311035156
pretrain_time: 82.87632393836975
start_inference_test
finish_inference_test
time_use: 0.019431114196777344
mse of tau: tensor(nan, device='cuda:0', grad_fn=<DivBackward0>)
mse of y_fact: tensor(9.6279, device='cuda:0', grad_fn=<DivBackward0>)


(tensor(nan, device='cuda:0', grad_fn=<DivBackward0>),
 tensor(9.6279, device='cuda:0', grad_fn=<DivBackward0>))